# Window Functions


## OVER()

In [ ]:
The Problem With GROUP BY

Consider:

employees
| emp_id | department | salary |
| ------ | ---------- | ------ |
| 1      | IT         | 90000  |
| 2      | IT         | 70000  |
| 3      | IT         | 50000  |
| 4      | HR         | 60000  |
| 5      | HR         | 40000  |

Suppose I ask:

Show every employee and also show the average salary of their department.

Most beginners try:

SELECT
    department,
    AVG(salary)
FROM employees
GROUP BY department;

Result:
| department | avg_salary |
| ---------- | ---------- |
| IT         | 70000      |
| HR         | 50000      |

But notice:

❌ We lost individual employees.

What We Want
| emp_id | department | salary | dept_avg_salary |
| ------ | ---------- | ------ | --------------- |
| 1      | IT         | 90000  | 70000           |
| 2      | IT         | 70000  | 70000           |
| 3      | IT         | 50000  | 70000           |
| 4      | HR         | 60000  | 50000           |
| 5      | HR         | 40000  | 50000           |

We want:

Individual rows preserved
Aggregate information attached

This is exactly what Window Functions do.

In [ ]:
First Window Function
AVG(salary) OVER (
    PARTITION BY department
)

Query:

SELECT
    emp_id,
    department,
    salary,
    AVG(salary) OVER (
        PARTITION BY department
    ) AS dept_avg_salary
FROM employees;

Result:
| emp_id | department | salary | dept_avg_salary |
| ------ | ---------- | ------ | --------------- |
| 1      | IT         | 90000  | 70000           |
| 2      | IT         | 70000  | 70000           |
| 3      | IT         | 50000  | 70000           |
| 4      | HR         | 60000  | 50000           |
| 5      | HR         | 40000  | 50000           |

Mental Model

Think:

GROUP BY

= Collapse rows

OVER()

= Keep rows and calculate something across them

This distinction is crucial.

## ROW_NUMBER()

In [ ]:
This is one of the most important SQL functions for Data Analysts.

What does ROW_NUMBER() do?

It assigns a unique sequential number to each row.

Example:
| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 2      | 50000  |
| 3      | 70000  |

Query:

SELECT
    emp_id,
    salary,
    ROW_NUMBER() OVER (
        ORDER BY salary DESC
    ) AS row_num
FROM employees;

Result:
| emp_id | salary | row_num |
| ------ | ------ | ------- |
| 1      | 100000 | 1       |
| 3      | 70000  | 2       |
| 2      | 50000  | 3       |


In [ ]:
Mental Model
ORDER BY salary DESC

Sorts:
| salary |
| ------ |
| 100000 |
| 70000  |
| 50000  |

Then:

ROW_NUMBER()

assigns:
1
2
3


In [ ]:
Why Analysts Use It
Find Top Earners
SELECT *
FROM (
    SELECT
        emp_id,
        salary,
        ROW_NUMBER() OVER (
            ORDER BY salary DESC
        ) AS rn
    FROM employees
) t
WHERE rn <= 3;

Top 3 highest-paid employees.

Very common interview question.

## PARTITION BY + ROW_NUMBER()

In [ ]:
Now it gets interesting.

Suppose:
| emp_id | department | salary |
| ------ | ---------- | ------ |
| 1      | IT         | 100000 |
| 2      | IT         | 50000  |
| 3      | IT         | 70000  |
| 4      | HR         | 60000  |
| 5      | HR         | 40000  |

Query:

SELECT
    emp_id,
    department,
    salary,
    ROW_NUMBER() OVER (
        PARTITION BY department
        ORDER BY salary DESC
    ) AS rn
FROM employees;

IT Department
Sort salaries:

| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 3      | 70000  |
| 2      | 50000  |

Assign:
1
2
3

HR Department
Sort salaries:

| emp_id | salary |
| ------ | ------ |
| 4      | 60000  |
| 5      | 40000  |

Assign:
1
2

Final Result
| emp_id | department | salary | rn |
| ------ | ---------- | ------ | -- |
| 1      | IT         | 100000 | 1  |
| 3      | IT         | 70000  | 2  |
| 2      | IT         | 50000  | 3  |
| 4      | HR         | 60000  | 1  |
| 5      | HR         | 40000  | 2  |

Notice:
The numbering restarts for each department.
That's what PARTITION BY does.


## RANK()

In [ ]:
Consider:
| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 2      | 100000 |
| 3      | 70000  |

Query:

SELECT
    emp_id,
    salary,
    ROW_NUMBER() OVER (
        ORDER BY salary DESC
    ) AS rn
FROM employees;

Result:
| emp_id | salary | rn |
| ------ | ------ | -- |
| 1      | 100000 | 1  |
| 2      | 100000 | 2  |
| 3      | 70000  | 3  |

Notice:

Both employees have the same salary.

Yet one got rank 1 and the other got rank 2.

Why?

Because ROW_NUMBER() doesn't care about ties.

It simply says:

"You are first."

"You are second."

"You are third."

Even if values are identical.



In [ ]:
Enter RANK()

Same data:
| emp_id | salary |
| ------ | ------ |
| 1      | 100000 |
| 2      | 100000 |
| 3      | 70000  |

Query:

SELECT
    emp_id,
    salary,
    RANK() OVER (
        ORDER BY salary DESC
    ) AS rnk
FROM employees;

Result:
| emp_id | salary | rnk |
| ------ | ------ | --- |
| 1      | 100000 | 1   |
| 2      | 100000 | 1   |
| 3      | 70000  | 3   |

Notice:
1
1
3

Rank 2 is skipped.
This is called a gap rank.

## Dense Rank()

In [ ]:
SELECT
    emp_id,
    salary,
    DENSE_RANK() OVER (
        ORDER BY salary DESC
    ) AS drnk
FROM employees;

Result:
| emp_id | salary | drnk |
| ------ | ------ | ---- |
| 1      | 100000 | 1    |
| 2      | 100000 | 1    |
| 3      | 70000  | 2    |

Notice:
1
1
2

No gap.